### About partition?

为什么需要按日期分区？它的设计意义是什么？
在大数据物理存储中，如果把一亿行数据不加区分地全倒进一个文件夹里，下游如果只想查“今天”的数据，计算引擎就必须把这一亿行文件全部从硬盘里读一遍、在内存里过滤一遍（全表扫描，Full Table Scan），I/O 成本高得吓人。

按日期分区（Partition By date） 的物理本质，是命令系统在底层云存储里，强行按日期建一堆独立的子文件夹：

e.g.
yuto_orders_partitioned/
  
  ├── order_date=2026-06-01/  ➔ 📁 里面只放6月1号的 Parquet 文件
  
  ├── order_date=2026-06-02/  ➔ 📁 里面只放6月2号的 Parquet 文件


当下游执行 WHERE order_date = '2026-06-01' 时，引擎拥有“分区裁剪（Partition Pruning）”的超能力——它会瞬间跳过其余几百个文件夹，直奔 6 月 1 号的子目录，I/O 开销直接暴跌 99%！


In [0]:
from pyspark.sql import functions as F

# 拉取bronze层的order_items表，然后创建一个testDate列，把全部日期，匀在2026年6月1日至6月9日期间

oi_test = spark.table("bronze_order_items").withColumn(
    "testDate",
    F.date_add(F.to_date(F.lit("2026-06-01")), F.floor(F.rand() * 10).cast("int"))
)


In [0]:
oi_test.write.format("delta") \
    .partitionBy("testDate") \
    .mode("overwrite") \
    .saveAsTable("silver_order_items_partitioned")

print("🎉 分区表silver_order_items_partitioned物理写入成功！")


### 优化文件大小Optimize & Compact
为什么写入时不加 coalesce，而要放在后面优化？
在写入前加 coalesce(1) 会直接反噬、阉割上游的 CPU 并发度。
大厂标准的生产姿势是：写入时让全网小兵火力全开地写，哪怕吐出一堆小文件碎碎片也没关系。写完之后，我们反手下达一行 Delta Lake 特有的管理型军令，在【存储层】无痛合并小文件！

```
spark.sql("OPTIMIZE table_A")
```

这是 Delta Lake 专属的优化命令，核心作用是：
对表table_A进行小文件合并（Compaction），将多个小 Parquet 文件合并为更大的文件，减少 I/O 次数，提升查询性能。
清理已删除 / 更新数据的旧版本，回收存储空间（配合 VACUUM 可彻底清理）。
对分区表（如你按 testDate 分区的表）会按分区分别优化，不会跨分区合并数据。

In [0]:
spark.sql("OPTIMIZE silver_order_items_partitioned")

print("⚡ 存储层小文件紧凑化优化（Compaction）完美收工！")

### Data Integrity Check数据完整性测试
证明，存储在 Delta 表里的数据，跟源头相比没有丢掉一分子，也没有产生任何重复

In [0]:
source_cnt = spark.table("bronze_order_items").count()
target_cnt = spark.table("silver_order_items_partitioned").count()

print(f"📊 [行数对账] 上游源头总行数: {source_cnt} 行 | 下游落盘总行数: {target_cnt} 行")
assert source_cnt == target_cnt, "🚨 警报！行数对不上，数据在落盘过程中发生丢失或翻倍！"
print("✅ 行数对账完美闭环！\n")

In [0]:
# 数额对账

source_sum = spark.table("bronze_order_items").agg(F.sum("price")).collect()[0][0]

target_sum = spark.table("silver_order_items_partitioned").agg(F.sum("price")).collect()[0][0]

print(f"💰 [金额对账] 上游财务总额: {source_sum:.2f} | 下游资产总额: {target_sum:.2f}")
# 浮点数对账，允许极微小的精度误差
assert abs(source_sum - target_sum) < 0.01, "🚨 警报！财务金额对不上，数据遭到物理污染！"
print("✅ 资产金额对账完美闭狂！\n")

In [0]:
partition_detail = spark.sql("SHOW PARTITIONS silver_order_items_partitioned").collect()

print(f"📂 [分区审计] 当前 Delta 表底层已被成功切割为 {len(partition_detail)} 个物理日期文件夹：")
for p in partition_detail:
    print(f"  └── 📁 {p[0]}")